In [ ]:
import pandas as pd

from os import path
import re

from pymodulon.io import load_json_model
from pymodulon.plotting import *

import numpy as np
import plotly.express as px

In [ ]:
imods = pd.read_csv('../../data/processed_data/iModulon_table_annotated.csv', index_col=0)
imods = imods.reset_index()

In [ ]:
imods.head()

In [ ]:
fig = px.treemap(imods, path=['category', 'name'], values='exp_var',
                  color='category', width=1200, height=2400,)
fig.update_traces(textinfo= 'label', textfont_size = 40)
fig.update_layout(margin = dict(t=10, l=10, r=10, b=10))

exp_var_tresh = .005

for i in range(len(fig.data[0]['labels'])):
    label = fig.data[0]['labels'][i]
    value = fig.data[0]['values'][i]
    if len(label.split()) > 0 and label not in ['Cellular Processes', 'Cell Cycle']:
        if value > exp_var_tresh and 'single_gene' not in label:
            fig.data[0]['labels'][i] = '<br>'.join(label.split())
        elif 'single_gene' in label:
            fig.data[0]['labels'][i] = 'SG ' + label.split('_')[-1]
        else:
            fig.data[0]['labels'][i] = ''
    
fig.show()

In [ ]:
fig.write_image("images/fig1_treemap.svg")

In [ ]:
import pandas as pd
import plotly.express as px

# Set absolute threshold
abs_thresh = 0.015

# Group small components into "Other" based on absolute exp_var
lumped = []

for cat, group in imods.groupby('category'):
    large = group[group['exp_var'] >= abs_thresh]
    small = group[group['exp_var'] < abs_thresh]
    
    lumped.append(large)
    
    if not small.empty:
        # Count how many small components were lumped
        count = len(small)
        other_label = f'{count} Other iModulons'
        
        other_row = pd.DataFrame({
            'category': [cat],
            'name': [other_label],
            'exp_var': [small['exp_var'].sum()]
        })
        lumped.append(other_row)

# Combine into a new DataFrame
imods_lumped = pd.concat(lumped, ignore_index=True)

# Create treemap
fig = px.treemap(
    imods_lumped,
    path=['category', 'name'],
    values='exp_var',
    color='category',
    width=1700,
    height=2400
)

# Format text
fig.update_traces(textinfo='label+value', textfont_size=80)

# Optional: Clean up labels
exp_var_tresh = 0.005
for i in range(len(fig.data[0]['labels'])):
    label = fig.data[0]['labels'][i]
    value = fig.data[0]['values'][i]

    if label.startswith('Other ('):
        continue  # Leave as is
    elif value > exp_var_tresh and 'single_gene' not in label:
        fig.data[0]['labels'][i] = '<br>'.join(label.split())
    elif 'single_gene' in label:
        fig.data[0]['labels'][i] = 'SG ' + label.split('_')[-1]
    else:
        fig.data[0]['labels'][i] = ''

# Layout settings
fig.update_layout(margin=dict(t=10, l=10, r=10, b=10))

# Show the figure
fig.show()


In [ ]:
fig.write_image("images/fig1_treemap_simplified.svg")

## Figure of Precision and Recall of Regulatory iModulons

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
DF_metadata = ica_data.imodulon_table[~ica_data.imodulon_table.regulator.isna()]
DF_metadata.regulator = DF_metadata.regulator.str.split('(').str[0]

In [ ]:
scatt_fig = px.scatter(DF_metadata, y="precision", x="recall",
                       color="category", width=1600, height=1000, size="imodulon_size", text=DF_metadata.index + "<br>Reg: " + DF_metadata.regulator)
scatt_fig.update_traces(marker=dict(line=dict(width=4,
                                        color='Black')),
                  selector=dict(mode='markers'))

In [ ]:
scatt_fig.write_image("images/fig1_regulator_scatter.svg")

# Scatterplot figure for strong regulons

In [ ]:
trn_enrichment = pd.read_csv('../../data/processed_data/trn_enrichments.csv')

In [ ]:
DF_plotting = ica_data.imodulon_table.copy()
DF_plotting.index.set_names('imodulon', inplace=True)
DF_plotting = DF_plotting.reset_index()[['imodulon', 'category', 'precision', 'recall', 'regulator', 'imodulon_size', 'regulon_size']]
DF_plotting = DF_plotting.dropna()

for i, row in trn_enrichment.loc[trn_enrichment.groupby('imodulon').qvalue.idxmin()].iterrows():
    DF_plotting.loc[row.imodulon] = [DF_plotting.loc[row.imodulon, 'imodulon'],
                                    DF_plotting.loc[row.imodulon, 'category'],
                                    row.precision,
                                    row.recall, 
                                    row.regulator,
                                    DF_plotting.loc[row.imodulon, 'imodulon_size'],
                                    row.regulon_size]


In [ ]:
scatt_fig = px.scatter(DF_plotting, y="precision", x="recall",
                       color="category", width=1600, height=1000, size="imodulon_size")
scatt_fig.update_traces(marker=dict(line=dict(width=4,
                                        color='Black')),
                  selector=dict(mode='markers'))

In [ ]:
scatt_fig.write_image("images/fig1_regulator_scatter_strong.svg")